# Tokenizer

SentencePiece BPE tokenizer used by the pretraining pipeline. Run this notebook directly in Colab to train, encode, and decode text interactively.

## Colab Bootstrap

Run this cell first. If you already ran `colab_setup.ipynb` in the same Colab runtime, this will reuse that setup; if not, it installs the small dependency set and clones the repo.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/markschatza/llm-from-scratch.git"
REPO_REF = "codex/colab-setup-workflow"
REPO_DIR = Path("/content/llm-from-scratch")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pretrain" / "tokenizer.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repo root. Run this notebook from inside the llm-from-scratch checkout."
    )

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "numpy", "torch"], check=True)
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
    project_root = REPO_DIR
else:
    project_root = find_project_root(Path.cwd())

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project root: {project_root}")
print(f"repo ref: {REPO_REF if IN_COLAB else 'local checkout'}")
print(f"in colab: {IN_COLAB}")

In [ ]:
from pathlib import Path

from pretrain.tokenizer import Tokenizer


## Quick Test

Set `CORPUS_PATH` to a text file in the repo or an uploaded Colab file, then train a small tokenizer and round-trip a sample.

In [ ]:
CORPUS_PATH = Path("data/van-life-story.txt")
OUTPUT_DIR = Path("pretrain/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tok = Tokenizer.train(CORPUS_PATH, vocab_size=1024, output_dir=OUTPUT_DIR, name="sp")
ids = tok.encode("Hello, world!")
print(ids)
print(tok.decode(ids))
